# CivicBrain - YOLOv8 training on the existing images (7-day MVP)

**How to use (human, about 10 minutes of clicking):**
1. Right panel → **Add Input** → your private dataset `civicbrain-yolo` (made from `kaggle_upload/civicbrain-yolo.zip`).
2. Right panel → **Settings**: Accelerator **GPU** (T4 x2 or P100), **Internet on**.
3. Run cells 1-4 one by one (**Run** button). Cell 4 is a 3-epoch timing run (~10 min) and runs only in this interactive session.
4. If cells 1-4 printed `OK`, click **Save Version** → **Save & Run All (Commit)** → Save.
   The full training (cell 5) runs on Kaggle in the background (2-4 h, also when you close the browser).
5. When the version is finished: open it → **Output** → download **`civicbrain_yolo_outputs.zip`** →
   put it in `C:\dev\civicbrain\kaggle_download\` → tell the agent "model downloaded" (prompt P08).

Nothing here changes your images or labels. Classes are FROZEN: 0 Pothole, 1 Garbage Accumulation, 2 Waterlogging, 3 Road Damage.
Metrics are on the existing test split - **not** a Talegaon field test (say so in the report).

In [ ]:
# Cell 1 - setup (pinned version = the one the CivicBrain worker uses)
import glob
import json
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.168"], check=True)
FORCE_RUN_TYPE = None   # leave None. Only if a committed run printed "interactive session": set "Batch" and commit again
RUN_TYPE = FORCE_RUN_TYPE or os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "Interactive")   # Kaggle: Interactive (editor) / Batch (commit)
print("run type:", RUN_TYPE)
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
print("GPU:", gpu.stdout.strip() or "NONE - turn on the GPU accelerator in Settings, then restart the session")
ultralytics = __import__("ultralytics")
print("ultralytics", ultralytics.__version__)
assert ultralytics.__version__ == "8.4.168", "wrong ultralytics version"
assert gpu.returncode == 0 and gpu.stdout.strip(), "no GPU"
print("OK")

In [ ]:
# Cell 2 - find the dataset and the training scripts (works for any dataset name / Kaggle input layout)
yaml_hits = [Path(p) for p in glob.glob("/kaggle/input/**/data.yaml", recursive=True)
             if (Path(p).parent / "images").is_dir()]
script_hits = [Path(p) for p in glob.glob("/kaggle/input/**/training/train_yolo_mvp.py", recursive=True)]
assert len(yaml_hits) == 1, f"expected exactly one yolo/data.yaml with images/ next to it, found: {yaml_hits}"
assert script_hits, "training/train_yolo_mvp.py not found in the dataset - rebuild the zip with make-kaggle-package.ps1"
DATA_ROOT = yaml_hits[0].parent
TRAIN_DIR = Path("/kaggle/working/training")
TRAIN_DIR.mkdir(parents=True, exist_ok=True)
for f in script_hits[0].parent.glob("*.py"):
    shutil.copy2(f, TRAIN_DIR / f.name)
for split in ("train", "val", "test"):
    n = sum(1 for p in (DATA_ROOT / "images" / split).rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"})
    print(f"{split:5s} images: {n}")
print("dataset root:", DATA_ROOT)
print("OK")

In [ ]:
# Cell 3 - check labels + leakage and build the oversampled training list (garbage x3, waterlogging x2)
MVP_DIR = Path("/kaggle/working/mvp")
r = subprocess.run([sys.executable, str(TRAIN_DIR / "prepare_mvp_dataset.py"), "--root", str(DATA_ROOT), "--out-dir", str(MVP_DIR)],
                   capture_output=True, text=True)
print(r.stdout[-6000:])
print(r.stderr[-3000:])
if r.returncode == 1:
    print("WARNING: label problems were found (see above / mvp/dataset_report.json). Bad lines are skipped by YOLO; "
          "fix them in data/yolo/labels if there are many, rebuild the zip and start again.")
assert r.returncode in (0, 1), "dataset structure error - read the message above"
assert (MVP_DIR / "data_mvp.yaml").is_file()
print("OK")

In [ ]:
# Cell 4 - 3-epoch timing run (interactive session only; skipped in the committed run)
if RUN_TYPE == "Interactive":
    r = subprocess.run([sys.executable, str(TRAIN_DIR / "train_yolo_mvp.py"), "--data", str(MVP_DIR / "data_mvp.yaml"),
                        "--out", "/kaggle/working/timing_out", "--timing"], capture_output=True, text=True)
    print(r.stdout[-4000:])
    print(r.stderr[-4000:])
    assert r.returncode == 0, "timing run failed - read the error above"
    print("OK - now click Save Version -> Save & Run All (Commit). Cell 5 trains only in the committed run.")
else:
    print("committed run - timing skipped")

In [ ]:
# Cell 5 - full training + test-split evaluation + ONNX export (committed run only)
OUT = Path("/kaggle/working/out")
if RUN_TYPE == "Batch":
    r = subprocess.run([sys.executable, str(TRAIN_DIR / "train_yolo_mvp.py"), "--data", str(MVP_DIR / "data_mvp.yaml"),
                        "--out", str(OUT), "--model", "yolov8s.pt", "--epochs", "120", "--patience", "25", "--imgsz", "640"])
    assert r.returncode == 0, "training failed - open the log of this version"
else:
    print("interactive session - full training runs only after Save Version -> Save & Run All (Commit)")

In [ ]:
# Cell 6 - one small zip to download: model, metrics, manifest entry, dataset report, key plots
if RUN_TYPE == "Batch":
    pick = [OUT / "yolov8s_civicbrain.onnx", OUT / "yolo_metrics.json", OUT / "manifest_entry.json", MVP_DIR / "dataset_report.json"]
    plots = [p for p in OUT.glob("runs/**/*") if p.name in {"confusion_matrix.png", "confusion_matrix_normalized.png", "results.png",
             "BoxPR_curve.png", "PR_curve.png", "results.csv"}]
    target = Path("/kaggle/working/civicbrain_yolo_outputs.zip")
    with zipfile.ZipFile(target, "w", compression=zipfile.ZIP_DEFLATED) as z:
        for p in pick:
            assert p.is_file(), f"missing {p}"
            z.write(p, p.name)
        for p in plots:
            z.write(p, "plots/" + p.relative_to(OUT / "runs").as_posix())
    print(json.dumps(json.loads((OUT / "yolo_metrics.json").read_text())["overall"], indent=2))
    print(f"DONE: download {target.name} from the Output tab ({target.stat().st_size / 1e6:.1f} MB)")